In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib as pl

In [ ]:
root_path = pl.Path.cwd().parent.parent
data_path = root_path / "data" 

In [ ]:
data_normalized = pd.read_csv(data_path/"processed"/"googleplaystore_normalized.csv")

In [ ]:
colunas_metricas = ['D_norm', 'A_norm', 'C_norm', 'U_norm', 'P_norm']

matriz_dados = data_normalized[colunas_metricas].copy()


epsilon = 1e-9
matriz_dados = matriz_dados + epsilon


soma_colunas = matriz_dados.sum(axis=0)
proporcoes = matriz_dados / soma_colunas
# print(proporcoes)

n_categorias = len(matriz_dados) 
k = 1.0 / np.log(n_categorias)   
entropia = -k * (proporcoes * np.log(proporcoes)).sum(axis=0)


grau_diversidade = 1.0 - entropia
pesos_otimizados = grau_diversidade / grau_diversidade.sum()


w_d = pesos_otimizados['D_norm']
w_r = pesos_otimizados['A_norm']
w_q = pesos_otimizados['C_norm']
w_t = pesos_otimizados['U_norm']
w_o = pesos_otimizados['P_norm']

print("="*50)
print("O algoritmo calculou automaticamente o peso de cada métrica:\n")
print(f"Demanda (D):      {w_d*100:05.2f}%")
print(f"Avaliação (A):    {w_r*100:05.2f}%")
print(f"Concorrência (C): {w_q*100:05.2f}%")
print(f"Estagnação (T):   {w_t*100:05.2f}%")
print(f"Polarização Score (P): {w_o*100:05.2f}%\n")


data_normalized['Disruption_Score'] = (
    (w_d * data_normalized['D_norm']) +
    (w_r * data_normalized['A_norm']) +
    (w_q * data_normalized['C_norm']) +
    (w_t * data_normalized['U_norm']) +
    (w_o * data_normalized['P_norm']) 
)

resultado_final = data_normalized.sort_values(by='Disruption_Score', ascending=False)

print("="*50)
print("Categorias ordenadas pelo Score de Oportunidade:\n")

print(resultado_final[["Category","C","A","D","U","Score_Oportunidade","Disruption_Score"]].round(4).to_string(index=False))



In [ ]:
plt.figure(figsize=(18, 6))
sns.barplot(data=resultado_final.head(5), x="Category", y="Disruption_Score", palette="magma")
plt.xticks(rotation=45, ha='right') 
plt.title("Ranking Estratégico: Score de Disrupção Otimizado por Entropia", fontsize=16, weight='bold')
plt.tight_layout()
plt.show()

In [ ]:

import matplotlib.pyplot as plt

pesos = [w_d, w_r, w_q, w_t, w_o]
cores = ['#2c3e50', '#16a085', '#2980b9', '#8e44ad', '#d35400']

labels_legenda = [
    f'Downloads (D): {w_d*100:.1f}%', 
    f'Avaliação (A): {w_r*100:.1f}%', 
    f'Concorrência (C): {w_q*100:.1f}%', 
    f'Estagnação (U): {w_t*100:.1f}%', 
    f'Polarização Score (P): {w_o*100:.1f}%'
]

destaque = (0, 0, 0, 0, 0.08) 

fig, ax = plt.subplots(figsize=(16, 8))

wedges, texts = ax.pie(
    pesos, 
    colors=cores, 
    startangle=140,
    explode=destaque, # Puxa a fatia laranja para fora
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

circulo_central = plt.Circle((0,0), 0.65, fc='white')
fig.gca().add_artist(circulo_central)

ax.legend(
    wedges, 
    labels_legenda,
    title="Peso Estratégico",
    title_fontproperties={'weight':'bold', 'size':13},
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1), # Empurra a legenda para fora do gráfico, à direita
    fontsize=20,
    frameon=False # Tira a borda quadrada feia da legenda
)


plt.title("O Veredicto do Algoritimo: Distribuição de Pesos (Entropia)", fontsize=22, weight='bold', pad=20)
plt.axis('equal') 
plt.tight_layout()
plt.show()